# EmotWen — Step 2: Supervised Fine-Tuning (SFT)

Two-stage SFT on `unsloth/Qwen3.5-0.8B`:
- **Stage 1** (tone): empathetic_dialogues + daily_dialog
- **Stage 2** (domain): synthetic journal conversations

Requires a free **Tesla T4** Colab instance.

**Prerequisites:** Run `01_data_prep.ipynb` first.

In [ ]:
# ── Install Unsloth + TRL stack ───────────────────────────────────────────────
%%capture
import os, importlib.util, sys
!pip install --upgrade -qqq uv
if importlib.util.find_spec('torch') is None or 'COLAB_' in ''.join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = 'numpy'; _pil = 'pillow'
    !uv pip install -qqq \
        'torch==2.8.0' 'triton>=3.3.0' {_numpy} {_pil} bitsandbytes 'xformers==0.0.32.post2' \
        'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo' \
        'unsloth[base] @ git+https://github.com/unslothai/unsloth'
elif importlib.util.find_spec('unsloth') is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers 'trl==0.22.2' unsloth unsloth_zoo
!uv pip install 'transformers==5.2.0'
!uv pip install --no-build-isolation flash-linear-attention 'causal_conv1d==1.6.0'
!uv pip install wandb datasets nltk

In [ ]:
# ── Mount Drive and set up repo ───────────────────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/emotwen-3.5-finetune'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/afonsomota/emotwen-3.5-finetune.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
except ImportError:
    REPO_DIR = os.path.dirname(os.getcwd())

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import nltk
nltk.download('punkt_tab', quiet=True)
print(f'Repo dir: {REPO_DIR}')

In [ ]:
# ── W&B login ─────────────────────────────────────────────────────────────────
import wandb
wandb.login()

In [ ]:
# ── Config overrides (edit this cell) ────────────────────────────────────────
CONFIG = {
    # W&B
    'wandb_project': 'emotwen-journal-chat',
    'run_name_s1': 'sft_stage1_v1',
    'run_name_s2': 'sft_stage2_v1',

    # Which stage(s) to run: '1', '2', or 'both'
    'stage': 'both',

    # Stage 1 steps (set lower for quick debug)
    # 'max_steps': 30,  # uncomment for quick test

    # Override output dirs to Drive for persistence
    # 'output_dir': '/content/drive/MyDrive/emotwen/outputs/sft_stage1',  # stage 1
}

In [ ]:
# ── Run SFT ───────────────────────────────────────────────────────────────────
from src.train_sft import run

results = run(CONFIG)
print('\n── SFT Results ──────────────────────────────────────────')
for k, v in results.items():
    print(f'  {k}: {v}')

In [ ]:
# ── GPU memory summary ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    used = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Peak GPU memory: {used:.2f} GB / {total:.2f} GB ({100*used/total:.1f}%)')